# 07 - Classification and evaluation

Fits LightGBM and Random Forest on every training-set scenario and scores all of
them on the same real hold-out set. CPU only. Run once over all datasets.
See `PIPELINE.md`, step 7.

In [ ]:
# Colab: upload df_train_*.csv, df_test_*.csv, *_pool.csv and smotenc_*.csv.
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    files.upload()

In [ ]:
%pip install -q lightgbm

In [ ]:
import glob
import itertools
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             recall_score, confusion_matrix, log_loss)

warnings.filterwarnings("ignore", category=UserWarning)
try:
    display
except NameError:
    display = print

## Config

In [ ]:
DATA_DIR    = "."
RESULTS_DIR = "."         # results_long.csv is kept here; a Drive path lets a cut session resume
DATASETS    = ["diabetes", "pageblocks", "yeast", "ecoli"]   # those without files are skipped

SEED = 42
CLASSIFIERS = ["lightgbm", "randomforest"]
REPEATS = {"lightgbm": 1, "randomforest": 3}   # LightGBM is deterministic here; RF is averaged

CURVE_MAX_ROWS = 20000    # stratified rows used to draw the loss curves
CURVE_STRIDE   = 5        # hold-out loss is evaluated every CURVE_STRIDE trees

POOL_PATTERNS = {
    "cwgan":     "cwgan_{ds}_*_pool.csv",
    "ecwgan":    "ecwgan_E12345_{ds}_*_pool.csv",
    "ecwganVAE": "ecwganVAE_{ds}_*_pool.csv",
}
SMOTE_PATTERN = "smotenc_{ds}_*.csv"

# Budget rule (Section 2.4)
IR_FULL_BALANCE   = 40    # IR >= this: every class to n_max
IR_NO_AUGMENT     = 10    # IR < this: no augmentation
RESIDUAL_IR       = 3     # in between: every class to n_max / RESIDUAL_IR
MIN_MODELLABLE    = 40    # a class with fewer real rows than this ...
MODELLABLE_FACTOR = 4     # ... is capped at MODELLABLE_FACTOR x its own count

## Data and files

In [ ]:
def find_one(pattern):
    hits = sorted(glob.glob(os.path.join(DATA_DIR, pattern)))
    if len(hits) > 1:
        raise ValueError(f"more than one file matches {pattern}, keep one: {hits}")
    return hits[0] if hits else None


def prepare(ds):
    df_train = pd.read_csv(os.path.join(DATA_DIR, f"df_train_{ds}.csv"))
    df_test = pd.read_csv(os.path.join(DATA_DIR, f"df_test_{ds}.csv"))
    target = df_train.columns[-1]
    df_test = df_test[df_train.columns]
    assert not df_train.isna().any().any() and not df_test.isna().any().any()

    h_train = set(pd.util.hash_pandas_object(df_train, index=False))
    overlap = int(pd.util.hash_pandas_object(df_test, index=False).isin(h_train).sum())
    assert overlap == 0, f"{ds}: {overlap} rows appear in both train and test"

    features = [c for c in df_train.columns if c != target]
    counts = df_train[target].value_counts().sort_index()
    pools = {g: find_one(p.format(ds=ds)) for g, p in POOL_PATTERNS.items()}
    ctx = dict(ds=ds, target=target, features=features, cols=list(df_train.columns),
               df_train=df_train, X_test=df_test[features], y_test=df_test[target],
               counts=counts, n_max=int(counts.max()),
               labels=sorted(set(df_train[target]) | set(df_test[target])),
               pools={g: p for g, p in pools.items() if p},
               smote=find_one(SMOTE_PATTERN.format(ds=ds)))
    print(f"{ds:11s} train {df_train.shape} | test {df_test.shape} | IR "
          f"{counts.max() / counts.min():.1f} | pools {list(ctx['pools'])} | "
          f"smotenc {'yes' if ctx['smote'] else 'no'}")
    return ctx


def load_synthetic(path, ctx):
    df = pd.read_csv(path)[ctx["cols"]]
    assert not df.isna().any().any(), f"{path} contains NaN"
    return df


DATASETS = [ds for ds in DATASETS
            if os.path.isfile(os.path.join(DATA_DIR, f"df_train_{ds}.csv"))
            and os.path.isfile(os.path.join(DATA_DIR, f"df_test_{ds}.csv"))]
CONTEXTS = {ds: prepare(ds) for ds in DATASETS}

## Training-set scenarios

Every generator scenario is a deterministic cut of the same pool (the first n rows
of each class).

In [ ]:
def derived_targets(counts, ir_full=IR_FULL_BALANCE, residual_ir=RESIDUAL_IR,
                    min_modellable=MIN_MODELLABLE):
    """Per-class target counts from the class counts alone (the budget rule)."""
    n_max, n_min = int(counts.max()), int(counts.min())
    ir = n_max / max(n_min, 1)
    if ir < IR_NO_AUGMENT:
        target = {c: int(counts[c]) for c in counts.index}
        rule = f"IR {ir:.1f} < {IR_NO_AUGMENT}: leave the data alone"
    elif ir >= ir_full:
        target = {c: n_max for c in counts.index}
        rule = f"IR {ir:.1f} >= {ir_full}: full balance"
    else:
        cap = int(np.ceil(n_max / residual_ir))
        target = {c: max(int(counts[c]), min(cap, n_max)) for c in counts.index}
        rule = f"{IR_NO_AUGMENT} <= IR {ir:.1f} < {ir_full}: residual IR {residual_ir}"
    floored = []
    for c in counts.index:
        n_c = int(counts[c])
        if n_c < min_modellable and target[c] > n_c * MODELLABLE_FACTOR:
            target[c] = n_c * MODELLABLE_FACTOR
            floored.append((c, n_c))
    return target, rule, floored


def quotas(ctx):
    """Pool rows taken per class by each scenario."""
    counts, n_max = ctx["counts"], ctx["n_max"]
    r3 = int(np.ceil(n_max / 3))
    derived, rule, _ = derived_targets(counts)
    return dict(
        topup={c: n_max - int(counts[c]) for c in counts.index if int(counts[c]) < n_max},
        full={c: n_max for c in counts.index},
        match={c: int(counts[c]) for c in counts.index},
        topup_r3={c: r3 - int(counts[c]) for c in counts.index if int(counts[c]) < r3},
        topup_derived={c: derived[c] - int(counts[c]) for c in counts.index
                       if derived[c] > int(counts[c])},
        full_derived=dict(derived), rule=rule)


def take(pool, per_class, ctx):
    parts = []
    for cls, n in per_class.items():
        p = pool[pool[ctx["target"]] == cls]
        if len(p) < n:
            raise ValueError(f"{ctx['ds']}: pool holds {len(p):,} rows of class {cls}, need {n:,}")
        parts.append(p.iloc[:n])
    return pd.concat(parts, ignore_index=True)


for ds in DATASETS:
    print(f"{ds:11s} budget rule: {quotas(CONTEXTS[ds])['rule']}")

## Classifiers and metrics

In [ ]:
def build(name, seed, balanced=False):
    """balanced=True is the cost-sensitive control."""
    w = "balanced" if balanced else None
    if name == "lightgbm":
        return lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  num_leaves=31, reg_alpha=0.1, reg_lambda=0.1,
                                  min_child_samples=20, random_state=seed, class_weight=w,
                                  verbosity=-1, n_jobs=-1)
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                  class_weight=w, random_state=seed, n_jobs=-1)


def g_mean(y_true, y_pred, labels):
    """Geometric mean of per-class recalls; a class never predicted gives 0."""
    r = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    return float(np.prod(r) ** (1 / len(r)))


def evaluate(y_true, y_pred, labels):
    m = {"accuracy": accuracy_score(y_true, y_pred),
         "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
         "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
         "g_mean": g_mean(y_true, y_pred, labels)}
    for lab, r in zip(labels, recall_score(y_true, y_pred, labels=labels,
                                           average=None, zero_division=0)):
        m[f"recall_class_{lab}"] = float(r)
    return m, confusion_matrix(y_true, y_pred, labels=labels)


def strat_index(y, n):
    if len(y) <= n:
        return y.index
    frac = n / len(y)
    return pd.concat([s.sample(max(1, int(round(len(s) * frac))), random_state=SEED)
                      for _, s in y.groupby(y)]).index


def fit_model(model, name, df_tr, ctx, record):
    """Fit; for LightGBM optionally record the loss per boosting iteration.
    eval_set holds training rows only; the hold-out curve is computed after fitting."""
    X, y = df_tr[ctx["features"]], df_tr[ctx["target"]]
    if not (record and name == "lightgbm"):
        model.fit(X, y)
        return None
    tr_idx = strat_index(y, CURVE_MAX_ROWS)
    history = {}
    model.fit(X, y, eval_set=[(X.loc[tr_idx], y.loc[tr_idx])], eval_names=["train"],
              callbacks=[lgb.record_evaluation(history), lgb.log_evaluation(0)])
    history["test"] = holdout_curve(model, ctx, history)
    return history


def holdout_curve(model, ctx, history):
    labels = ctx["labels"]
    te_idx = strat_index(ctx["y_test"], CURVE_MAX_ROWS)
    X, y = ctx["X_test"].loc[te_idx], ctx["y_test"].loc[te_idx]
    metric = next(iter(history["train"]))
    total = model.n_estimators_ if hasattr(model, "n_estimators_") else model.n_estimators
    steps = sorted(set(list(range(1, total + 1, CURVE_STRIDE)) + [total]))
    seen = {c: i for i, c in enumerate(model.classes_)}
    out = []
    for k in steps:
        p = model.predict_proba(X, num_iteration=k)
        q = np.zeros((len(X), len(labels)))
        for j, lab in enumerate(labels):
            if lab in seen:
                q[:, j] = p[:, seen[lab]]
        out.append(float(log_loss(y, q, labels=labels)))
    return {metric: out, "_iterations": steps}


def curve_records(history, ds, label, clf):
    out = []
    for split, metrics in (history or {}).items():
        steps = metrics.get("_iterations")
        for metric, values in metrics.items():
            if metric == "_iterations":
                continue
            for it, v in zip(steps or range(1, len(values) + 1), values):
                out.append({"dataset": ds, "scenario": label, "classifier": clf,
                            "split": split, "metric": metric,
                            "iteration": int(it), "loss": float(v)})
    return out

## Run

Every finished scenario is appended to `results_long.csv` and skipped on the
next run, so an interrupted session resumes where it stopped.

In [ ]:
if str(RESULTS_DIR).startswith("/content/drive"):
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
os.makedirs(RESULTS_DIR, exist_ok=True)
LONG_FILE = os.path.join(RESULTS_DIR, "results_long.csv")
CURVE_FILE = os.path.join(RESULTS_DIR, "results_curves.csv")
CM_FILE    = os.path.join(RESULTS_DIR, "results_confusion.csv")


def read_or_empty(path):
    return pd.read_csv(path) if os.path.isfile(path) else pd.DataFrame()


previous, prev_curves, prev_cm = map(read_or_empty, (LONG_FILE, CURVE_FILE, CM_FILE))
DONE = (set(zip(previous["dataset"], previous["scenario"], previous["classifier"],
                previous["repeat"])) if len(previous) else set())
rows, curve_rows, cm_rows = [], [], []


def save_table(new_rows, earlier, path, keys):
    out = pd.concat([earlier, pd.DataFrame(new_rows)], ignore_index=True) if new_rows else earlier
    if len(out):
        out = out.drop_duplicates(subset=keys, keep="last")
        out.to_csv(path, index=False)
    return out


def save_all():
    return (save_table(rows, previous, LONG_FILE, ["dataset", "scenario", "classifier", "repeat"]),
            save_table(curve_rows, prev_curves, CURVE_FILE,
                       ["dataset", "scenario", "classifier", "split", "metric", "iteration"]),
            save_table(cm_rows, prev_cm, CM_FILE,
                       ["dataset", "scenario", "classifier", "true", "predicted"]))


def run_scenario(ctx, label, df_tr, n_synth, diagnose=False):
    ds, labels = ctx["ds"], ctx["labels"]
    todo = [(clf, rep) for clf in CLASSIFIERS for rep in range(REPEATS[clf])
            if (ds, label, clf, rep) not in DONE]
    if not todo:
        return
    for clf, rep in todo:
        model = build(clf, SEED + rep, balanced=(label == "class_weight"))
        t0 = time.time()
        history = fit_model(model, clf, df_tr, ctx, record=(diagnose and rep == 0))
        fit_s = time.time() - t0
        m, cm = evaluate(ctx["y_test"], model.predict(ctx["X_test"]), labels)
        rows.append({"dataset": ds, "scenario": label, "classifier": clf, "repeat": rep,
                     "n_train": len(df_tr), "n_synthetic": n_synth,
                     "fit_seconds": round(fit_s, 2), **m})
        DONE.add((ds, label, clf, rep))
        curve_rows.extend(curve_records(history, ds, label, clf))
        if rep == 0:
            for i, true in enumerate(labels):
                for j, pred in enumerate(labels):
                    cm_rows.append({"dataset": ds, "scenario": label, "classifier": clf,
                                    "true": true, "predicted": pred, "n": int(cm[i, j])})
    save_all()
    print(f"  done: {label:22s} ({len(df_tr):,} rows)")


for ds in sorted(DATASETS, key=lambda d: len(CONTEXTS[d]["df_train"])):
    ctx, q = CONTEXTS[ds], quotas(CONTEXTS[ds])
    df_train = ctx["df_train"]
    print(f"\n=== {ds} ===")

    run_scenario(ctx, "baseline", df_train, 0, diagnose=True)
    run_scenario(ctx, "class_weight", df_train, 0, diagnose=True)

    if ctx["smote"]:
        df_aug = load_synthetic(ctx["smote"], ctx)
        run_scenario(ctx, "smotenc", df_aug, len(df_aug) - len(df_train))
        del df_aug

    for g, path in ctx["pools"].items():
        pool = load_synthetic(path, ctx)
        aug = lambda quota: pd.concat([df_train, take(pool, quota, ctx)], ignore_index=True)

        run_scenario(ctx, g, aug(q["topup"]), sum(q["topup"].values()), diagnose=True)
        run_scenario(ctx, f"{g}_syn", take(pool, q["full"], ctx), sum(q["full"].values()))
        run_scenario(ctx, f"{g}_synmatch", take(pool, q["match"], ctx), sum(q["match"].values()))
        run_scenario(ctx, f"{g}_ratio3", aug(q["topup_r3"]), sum(q["topup_r3"].values()))

        # Budget rule: skipped where it coincides with an arm already fitted.
        if q["topup_derived"] != q["topup"]:
            df_derived = aug(q["topup_derived"]) if q["topup_derived"] else df_train
            run_scenario(ctx, f"{g}_derived", df_derived, sum(q["topup_derived"].values()))
        if q["full_derived"] not in (q["full"], q["match"]):
            run_scenario(ctx, f"{g}_dsyn", take(pool, q["full_derived"], ctx),
                         sum(q["full_derived"].values()))
        del pool

long_df, curves_df, confusion_df = save_all()
print(f"\n{len(long_df)} fits recorded, {long_df['scenario'].nunique()} scenarios")

## Budget-rule sensitivity (no refitting)

What the rule selects for each dataset over 60 threshold combinations.

In [ ]:
sens = []
for f, r, m in itertools.product([20, 30, 40, 50, 80], [2, 3, 5], [0, 20, 40, 80]):
    for ds in DATASETS:
        target, rule, floored = derived_targets(CONTEXTS[ds]["counts"], f, r, m)
        sens.append({"IR_FULL_BALANCE": f, "RESIDUAL_IR": r, "MIN_MODELLABLE": m,
                     "dataset": ds, "regime": rule.split(":")[1].strip(),
                     "rows": sum(target.values()), "classes floored": len(floored)})
sens = pd.DataFrame(sens)
for ds in DATASETS:
    chosen = sens[sens.dataset == ds]["regime"].value_counts()
    print(f"{ds:11s}", ", ".join(f"{k} ({n})" for k, n in chosen.items()))

## Results

In [ ]:
METRICS = ["balanced_accuracy", "f1_macro", "g_mean", "accuracy"]
agg = long_df.groupby(["dataset", "classifier", "scenario"])[METRICS].agg(["mean", "std"]).round(4)
agg.columns = [m if s == "mean" else f"{m}_sd" for m, s in agg.columns]
summary = agg.reset_index().sort_values(["dataset", "classifier", "balanced_accuracy"],
                                        ascending=[True, True, False])
display(summary[["dataset", "classifier", "scenario"] + METRICS])

In [ ]:
# Augmenting, dropping and replacing the real data (Section 3.2).
ba = summary.set_index(["dataset", "classifier", "scenario"])["balanced_accuracy"]
lines = []
for ds in sorted(long_df["dataset"].unique()):
    for g in POOL_PATTERNS:
        for clf in CLASSIFIERS:
            try:
                b, aug_, syn, match = (ba[(ds, clf, s)] for s in
                                       ("baseline", g, f"{g}_syn", f"{g}_synmatch"))
            except KeyError:
                continue
            lines.append({"dataset": ds, "classifier": clf, "generator": g, "baseline": b,
                          "real+synth": aug_, "synth only": syn, "synth matched": match,
                          "gain from augmenting": aug_ - b, "cost of dropping real": syn - aug_,
                          "cost of replacing real": match - b})
display(pd.DataFrame(lines).round(4))

In [ ]:
recall_cols = [c for c in long_df.columns if c.startswith("recall_class_")]
per_class = (long_df.groupby(["dataset", "classifier", "scenario"])[recall_cols]
             .mean().round(4))

# Per-class recall heatmap, LightGBM (colour version of Figure 3)
for ds in sorted(long_df["dataset"].unique()):
    block = per_class.loc[(ds, "lightgbm")].dropna(axis=1, how="all")
    cols = list(block.columns)
    fig, ax = plt.subplots(figsize=(1.5 * len(cols) + 4, 0.55 * len(block) + 2.5))
    im = ax.imshow(block.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels([c.replace("recall_class_", "class ") for c in cols], rotation=45, ha="right")
    ax.set_yticks(range(len(block))); ax.set_yticklabels(block.index)
    for i in range(block.shape[0]):
        for j in range(block.shape[1]):
            v = block.values[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                    color="black" if 0.25 < v < 0.75 else "white")
    ax.set_title(f"{ds} - per-class recall (lightgbm)")
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout(); plt.show()

## Loss curves (mentioned in Section 3.5, not shown in the paper)

In [ ]:
ORDER = ["baseline", "class_weight", "cwgan", "ecwgan", "ecwganVAE"]
for ds in sorted(curves_df["dataset"].unique()):
    block = curves_df[curves_df["dataset"] == ds]
    scenarios = [s for s in ORDER if s in set(block["scenario"])]
    ncol = min(4, len(scenarios))
    nrow = int(np.ceil(len(scenarios) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.0 * ncol, 3.4 * nrow), squeeze=False, sharey=True)
    for idx, (ax, sc) in enumerate(zip(axes.ravel(), scenarios)):
        part = block[block["scenario"] == sc]
        for split, style in (("train", "-"), ("test", "--")):
            line = part[part["split"] == split].sort_values("iteration")
            ax.plot(line["iteration"], line["loss"], style, label=split)
        ax.set_title(sc, fontsize=10); ax.set_xlabel("boosting iteration")
        if idx % ncol == 0:
            ax.set_ylabel(part["metric"].iloc[0])
        ax.grid(alpha=0.3); ax.legend(fontsize=8)
    for ax in axes.ravel()[len(scenarios):]:
        ax.axis("off")
    fig.suptitle(f"{ds} - LightGBM loss per iteration", y=1.02)
    plt.tight_layout(); plt.show()

## Confusion matrices (every scenario; Figure 4 shows the LightGBM leaders)

In [ ]:
for ds in sorted(confusion_df["dataset"].unique()):
    labels = CONTEXTS[ds]["labels"]
    for clf in CLASSIFIERS:
        block = confusion_df[(confusion_df.dataset == ds) & (confusion_df.classifier == clf)]
        scenarios = sorted(block["scenario"].unique())
        ncol = min(3, len(scenarios))
        nrow = int(np.ceil(len(scenarios) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 3.3 * nrow), squeeze=False)
        for ax, sc in zip(axes.ravel(), scenarios):
            m = (block[block.scenario == sc]
                 .pivot_table(index="true", columns="predicted", values="n", aggfunc="sum")
                 .reindex(index=labels, columns=labels, fill_value=0).to_numpy())
            share = m / np.maximum(m.sum(axis=1, keepdims=True), 1)
            ax.imshow(share, cmap="Blues", vmin=0, vmax=1)
            for i in range(len(labels)):
                for j in range(len(labels)):
                    ax.text(j, i, f"{int(m[i, j]):,}", ha="center", va="center", fontsize=7.5,
                            color="white" if share[i, j] > 0.5 else "black")
            ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=8)
            ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
            ax.set_xlabel("predicted", fontsize=8); ax.set_ylabel("true", fontsize=8)
            ax.set_title(sc, fontsize=9)
        for ax in axes.ravel()[len(scenarios):]:
            ax.axis("off")
        fig.suptitle(f"{ds} - confusion matrices ({clf}, seed 0)", y=1.02)
        plt.tight_layout(); plt.show()

## Export

In [ ]:
written = [LONG_FILE, CURVE_FILE, CM_FILE]
for name, df in [("results_summary.csv", summary),
                 ("results_per_class_recall.csv", per_class.reset_index()),
                 ("budget_rule_sensitivity.csv", sens)]:
    path = os.path.join(RESULTS_DIR, name)
    df.to_csv(path, index=False)
    written.append(path)
print("written:", *written, sep="\n  ")

if IN_COLAB and not str(RESULTS_DIR).startswith("/content/drive"):
    for path in written:
        files.download(path)